# Activity #1 — SOLUTION

**Workshop:** From Black Boxes to Glass Boxes  
**Instructor:** Cagri Temel — Hezarfen LLC

*This is the complete reference solution. Use only after attempting the student notebook.*


In [ ]:
!pip install -q neural-trees pandas matplotlib seaborn scikit-learn

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, confusion_matrix
from neural_trees import SoftDecisionTree

print('All set.')

In [ ]:
BASE = 'https://raw.githubusercontent.com/cgrtml/wsu-workshop-may15/main/data'
!wget -q -nc {BASE}/train_FD001.txt
!wget -q -nc {BASE}/test_FD001.txt
!wget -q -nc {BASE}/RUL_FD001.txt

In [ ]:
cols = ['engine_id', 'cycle'] + [f'op{i}' for i in range(1, 4)] + [f's{i}' for i in range(1, 22)]
train = pd.read_csv('train_FD001.txt', sep=r'\s+', header=None, names=cols)

max_cycles = train.groupby('engine_id')['cycle'].max().rename('max_cycle')
train = train.merge(max_cycles, on='engine_id')
train['RUL'] = (train['max_cycle'] - train['cycle']).clip(upper=125)

def bin_rul(rul):
    if rul < 30: return 0
    if rul < 80: return 1
    return 2

train['health'] = train['RUL'].apply(bin_rul)

### Solution to TODO 1 — plot engine 24

In [ ]:
engine = train[train['engine_id'] == 24]
fig, axes = plt.subplots(2, 2, figsize=(12, 6))
for ax, sensor in zip(axes.flat, ['s2', 's7', 's11', 's14']):
    ax.plot(engine['cycle'], engine[sensor])
    ax.set_title(f'Engine 24 — {sensor}')
    ax.set_xlabel('cycle')
plt.tight_layout(); plt.show()

# Same direction as engine 1: s2 and s11 trend up (temperature/pressure rising as engine wears),
# s7 trends down (HPC outlet pressure dropping as compressor degrades).

In [ ]:
all_sensors = [f's{i}' for i in range(1, 22)]
informative = [s for s in all_sensors if train[s].std() > 0.001]
X = StandardScaler().fit_transform(train[informative].values)
y = train['health'].values

### Solution to TODO 2 — stratified split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)
print(X_train.shape, X_test.shape)

### Solution to TODO 3 — train and score

In [ ]:
tree = SoftDecisionTree(
    depth=4,
    max_epochs=30,
    learning_rate=0.01,
    penalty_coef=1e-3,
    verbose=True,
)
tree.fit(X_train, y_train)
acc = (tree.predict(X_test) == y_test).mean()
print(f'Test accuracy: {acc:.3f}')

In [ ]:
y_pred = tree.predict(X_test)
labels = ['Critical', 'Caution', 'Healthy']

cm = confusion_matrix(y_test, y_pred)
fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=labels, yticklabels=labels, ax=ax)
ax.set_xlabel('Predicted'); ax.set_ylabel('True')
plt.title('Confusion matrix'); plt.show()

print(classification_report(y_test, y_pred, target_names=labels))

### Solution to TODO 4 — traverse one prediction

In [ ]:
splits = tree.get_split_weights()

idx = 17
x = X_test[idx]
true = labels[y_test[idx]]
pred_class = labels[tree.predict([x])[0]]
proba = tree.predict_proba([x])[0]
root_dominant = informative[np.argmax(np.abs(splits[0]))]

print(f'True class:      {true}')
print(f'Predicted class: {pred_class}')
print(f'Probabilities:   Critical={proba[0]:.2f}  Caution={proba[1]:.2f}  Healthy={proba[2]:.2f}')
print(f'Root node looks at: {root_dominant}')

# Walk down the dominant sensor at each level
print('\n=== Decision path ===')
for level, w in enumerate(splits):
    feat = np.argmax(np.abs(w))
    print(f'Level {level}: dominant sensor = {informative[feat]:4s}  (w = {w[feat]:+.3f})')